# Day 8 Project: Support-Ticket Classifier

## What You're Building

You run this notebook and it classifies a set of 10 sample support tickets into
categories (`Billing`, `Technical`, `Account`, `Feature Request`, `Other`),
printing for each ticket: the original text, the assigned label, a confidence
score, and a one-sentence reasoning. The final cell prints a summary table
showing how many tickets fell into each category.

That's the deliverable: 10 tickets classified, results printed, summary shown.
Runs entirely with Ollama (`llama3.2`) at `localhost:11434` — no paid API.

**Concepts this project composes:**
- **Lesson 1** — Zero-shot classification with a constrained-output prompt
- **Lesson 2** — The `CLASSIFY_SYSTEM_PROMPT` constant pattern and label validation
- **Lesson 3** — `ClassificationResult` Pydantic schema (label + reasoning + confidence)
- **Lesson 4** — Label validation: normalising model output to canonical spelling and falling back to "Other" when the label is out-of-set
- **Lesson 5** — `classify_batch`: error-safe loop with Day-5 logging

> Complete exercises 1–5 before starting this project.

## Setup

Imports and constants go here. The label set is fixed: you classify every ticket
into exactly one of these five categories.

In [ ]:
import json
import logging
import ollama
from pydantic import BaseModel, Field, ValidationError

# Configure logging once here, at the entry point (not inside functions)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)s  %(message)s",
)
logger = logging.getLogger(__name__)

MODEL = "llama3.2"

# The fixed label set — every ticket must land in exactly one of these
LABELS = ["Billing", "Technical", "Account", "Feature Request", "Other"]

# Ten sample support tickets to classify
TICKETS = [
    "I was charged twice for my subscription this month and need a refund.",
    "The app keeps crashing every time I try to open the settings screen.",
    "I forgot my password and the reset email is not arriving in my inbox.",
    "It would be amazing if you could add a dark mode to the dashboard.",
    "My invoice shows a different amount than what I agreed to in the contract.",
    "The API is returning 500 errors intermittently on the /users endpoint.",
    "I need to update the email address associated with my account.",
    "Could you add the ability to export reports as CSV? That would save us hours.",
    "Just wanted to say thank you — your support team resolved my issue super fast!",
    "My two-factor authentication codes are not working and I am locked out.",
]

## Step 1: Define the ClassificationResult Schema

Define a Pydantic `BaseModel` called `ClassificationResult` with three fields:
- `label: str` — the chosen category (must be one of the five allowed labels)
- `reasoning: str` — one sentence explaining why this label was chosen
- `confidence: float` — a self-reported score from 0.0 (uncertain) to 1.0 (certain)

This is the Day-4 Pydantic pattern applied to classification.

In [ ]:
# TODO: define ClassificationResult as a Pydantic BaseModel
# Fields: label (str), reasoning (str), confidence (float)
# Add Field(description=...) to each so the schema is self-documenting

class ClassificationResult(BaseModel):
    pass  # TODO: replace with real fields

## Step 2: Write the System Prompt Constant

Define `CLASSIFY_SYSTEM_PROMPT` as a module-level triple-quoted string.
It must:
- List all allowed labels (inject them via `{labels}` placeholder)
- Embed the JSON schema so the model knows what structure to return
- Constrain the output: one JSON object, no prose, no explanation outside the JSON
- Include a concrete example output line (the Day-4 schema-echo fix)

Use `json.dumps(ClassificationResult.model_json_schema(), indent=2)` to
get the schema string.

In [ ]:
# TODO: define CLASSIFY_SYSTEM_PROMPT as a module-level constant
# It should be a triple-quoted string with {labels} and {schema} placeholders
# Include an example JSON output to prevent schema-echo failures

CLASSIFY_SYSTEM_PROMPT = """TODO"""  # replace this

## Step 3: Implement `classify_ticket`

Write a function `classify_ticket(text, labels, model)` that:
1. Formats `CLASSIFY_SYSTEM_PROMPT` with the label list and schema string
2. Calls `ollama.chat()` with `format="json"` (JSON mode)
3. Parses the response with `ClassificationResult.model_validate_json()`
4. Returns the `ClassificationResult` on success, or `None` on `ValidationError`
   (log the failure — don't let it crash)
5. After parsing, validates that `result.label` is actually in `labels`;
   if not, log a warning and fix it to `"Other"`

In [ ]:
def classify_ticket(
    text: str,
    labels: list[str] = LABELS,
    model: str = MODEL,
) -> ClassificationResult | None:
    """Classify a single support ticket.

    Args:
        text:   The ticket text to classify.
        labels: The allowed label set.
        model:  Ollama model name.

    Returns:
        A ClassificationResult with label, reasoning, and confidence,
        or None if JSON parsing fails.
    """
    # TODO: Step 1 — format the system prompt
    # TODO: Step 2 — call ollama.chat with format="json"
    # TODO: Step 3 — parse with ClassificationResult.model_validate_json()
    # TODO: Step 4 — catch ValidationError, log it, return None
    # TODO: Step 5 — validate result.label is in labels; fix to "Other" if not
    pass

## Step 4: Implement `classify_batch`

Write `classify_batch(texts, labels, model)` that:
- Loops over `texts` with `enumerate` (Day-5 pattern)
- Calls `classify_ticket()` per item inside a `try/except`
- Records every outcome as a dict with keys `text`, `label`, `confidence`,
  `reasoning`, `status` (`"ok"` or `"error"`), `error`
- Logs: `INFO` at batch start and end, `DEBUG` per item, `WARNING` on failure
- Never stops the loop on a single failure

In [ ]:
def classify_batch(
    texts: list[str],
    labels: list[str] = LABELS,
    model: str = MODEL,
) -> list[dict]:
    """Classify a list of ticket texts, never stopping on a single failure.

    Args:
        texts:  List of ticket strings to classify.
        labels: The allowed label set.
        model:  Ollama model name.

    Returns:
        A list of dicts, one per input. Keys:
        'text', 'label', 'confidence', 'reasoning', 'status', 'error'.
    """
    # TODO: log batch start (how many items, which model)
    # TODO: loop with enumerate; log DEBUG per item
    # TODO: call classify_ticket inside try/except
    # TODO: on success append dict with status="ok" and all result fields
    # TODO: on failure/None append dict with status="error" and error message
    # TODO: log WARNING on failure
    # TODO: log batch summary (succeeded / failed counts) at end
    pass

## Step 5: Run the Classifier

Call `classify_batch(TICKETS, LABELS)` and print each result.
For every ticket print: the truncated ticket text, the assigned label,
the confidence score, and the one-sentence reasoning.

In [ ]:
# Run the batch classifier
results = classify_batch(TICKETS, LABELS)

# TODO: loop over results and print each ticket's classification
# Format: ticket number, label, confidence, reasoning, then the ticket text
print("\n=== Classification Results ===")
for i, r in enumerate(results, start=1):
    pass  # TODO: print the result

## Step 6: Summary Table

Count how many tickets landed in each category and print a summary table.
Use a dict comprehension or `collections.Counter`.

In [ ]:
# TODO: count tickets per label and print a summary table
# Show each label and its count; also show how many failed (status == "error")
print("\n=== Summary Table ===")
print(f"{'Category':<20} {'Count':>5}")
print("-" * 27)
# TODO: fill in the counts per label
# TODO: print failed count